# portfolio_rl end-to-end demo

This notebook is a guided smoke test for the `portfolio_rl` package. It walks through a compact research workflow:

1. Define a small asset universe.
2. Pretrain the sequence encoder on train-only data.
3. Train an RL policy on the resulting state representation.
4. Run deterministic evaluation.
5. Run Monte Carlo robustness evaluation.

The notebook is intentionally small enough to debug the pipeline, not to produce production-grade investment results.

## Before you run this notebook

Expected prerequisites:

- The dependencies from `requirements.txt` are installed.
- Internet access is available for `yfinance` downloads.
- You are running from the repository root so `portfolio_rl` is importable.

Expected outcome:

- Imports should succeed without modification.
- If market data cannot be downloaded, the first data-dependent step will fail with a fetch error rather than a modeling error.

In [ ]:
from pathlib import Path
from pprint import pprint

import numpy as np
import torch

from portfolio_rl.encoding.asset_encoder.autoencoder_trainer import AutoencoderTrainingConfig
from portfolio_rl.fusion.state_builder import AssetSpec
from portfolio_rl.training.pretrain_encoder import PretrainEncoderConfig, run_pretraining
from portfolio_rl.training.train_agent import TrainAgentConfig, run_training

ROOT = Path.cwd()
ARTIFACTS = ROOT / "artifacts" / "notebook_demo"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

np.random.seed(42)
torch.manual_seed(42)

print("Repository root:", ROOT)
print("Artifact directory:", ARTIFACTS)

## Step 1: choose a small asset universe and time splits

We use a compact universe to keep the smoke test manageable. The train, validation, and evaluation periods are strictly separated.

Expected outcome:

- The asset list should be short and interpretable.
- The date ranges should be sequential with no overlap between fit phases and held-out evaluation.
- The artifact paths should point to new files inside `artifacts/notebook_demo/`.

In [ ]:
asset_specs = [
    AssetSpec(ticker="SPY", asset_type="equity"),
    AssetSpec(ticker="QQQ", asset_type="equity"),
    AssetSpec(ticker="GLD", asset_type="commodity"),
]

train_start = "2018-01-01"
train_end = "2021-12-31"
val_start = "2022-01-01"
val_end = "2022-12-31"
eval_start = "2023-01-01"
eval_end = "2023-12-31"

encoder_checkpoint = ARTIFACTS / "encoder.pt"
policy_checkpoint = ARTIFACTS / "policy.pt"

print("Assets:", [asset.ticker for asset in asset_specs])
print("Encoder checkpoint:", encoder_checkpoint)
print("Policy checkpoint:", policy_checkpoint)

## Step 2: pretrain the LSTM autoencoder

This stage learns per-asset temporal representations from train-only normalized sequences. Validation loss is used for early stopping.

Expected outcome:

- `num_train_sequences` and `num_val_sequences` should both be positive.
- `history["train_loss"]` and `history["val_loss"]` should contain a short loss trace.
- The encoder checkpoint should be written to disk.

If this step fails, common causes are missing dependencies, network issues while downloading prices, or time windows that are too short for the chosen `window_size`.

In [ ]:
pretrain_config = PretrainEncoderConfig(
    asset_specs=asset_specs,
    train_start=train_start,
    train_end=train_end,
    val_start=val_start,
    val_end=val_end,
    window_size=32,
    hidden_dim=64,
    latent_dim=16,
    normalizer_window=20,
    checkpoint_path=str(encoder_checkpoint),
    trainer=AutoencoderTrainingConfig(
        batch_size=64,
        learning_rate=1e-3,
        max_epochs=5,
        patience=2,
        device="cpu",
    ),
)

pretrain_result = run_pretraining(pretrain_config)
pprint({
    "checkpoint_path": pretrain_result["checkpoint_path"],
    "num_train_sequences": pretrain_result["num_train_sequences"],
    "num_val_sequences": pretrain_result["num_val_sequences"],
    "train_loss_history": pretrain_result["history"]["train_loss"],
    "val_loss_history": pretrain_result["history"]["val_loss"],
})

## Step 3: train the RL agent and run evaluation

This stage loads the frozen encoder, builds the portfolio environment, trains the selected RL policy, runs deterministic held-out evaluation, and then runs Monte Carlo robustness evaluation.

Expected outcome:

- `episode_logs` should contain one entry per training episode.
- `evaluation` should report deterministic test metrics such as Sharpe, total return, volatility, drawdown, turnover, and final value.
- `monte_carlo_evaluation` should include both a per-simulation list and a summary with confidence-style percentile ranges.

For a quick smoke test, the number of episodes and simulations are intentionally small. For a more meaningful experiment, increase both after the pipeline works end to end.

In [ ]:
train_config = TrainAgentConfig(
    asset_specs=asset_specs,
    encoder_checkpoint=str(encoder_checkpoint),
    train_start=train_start,
    train_end=train_end,
    eval_start=eval_start,
    eval_end=eval_end,
    algo="ppo",
    window_size=32,
    normalizer_window=20,
    latent_dim=16,
    checkpoint_path=str(policy_checkpoint),
    num_episodes=3,
    monte_carlo_num_simulations=10,
    monte_carlo_confidence_level=0.95,
    monte_carlo_seed=42,
    device="cpu",
)

training_result = run_training(train_config)

print("Deterministic evaluation:")
pprint(training_result["evaluation"])

print("\nMonte Carlo summary:")
pprint(training_result["monte_carlo_evaluation"]["summary"])

## Step 4: inspect training episodes

Episode logs help verify that the training loop itself is functioning. In a short smoke test, the absolute numbers are less important than structural sanity.

Expected outcome:

- Each episode dictionary should contain return, Sharpe, loss, and update counts.
- Loss values should be finite.
- Returns may be noisy, especially with only a few episodes, but they should not all be missing or nonsensical.

In [ ]:
for row in training_result["episode_logs"]:
    pprint(row)

## Step 5: inspect individual Monte Carlo simulations

The deterministic evaluation shows the policy's mean-action behavior, while Monte Carlo rollouts show how sensitive outcomes are to stochastic action sampling.

Expected outcome:

- The simulation records should vary somewhat when the policy is stochastic.
- If all simulations are almost identical, the policy may be highly concentrated around one action.
- If simulations vary wildly, that can indicate an unstable or poorly calibrated policy.

In [ ]:
training_result["monte_carlo_evaluation"]["simulations"][:3]

## Step 6: interpret the outputs

A healthy end-to-end test usually looks like this:

- The encoder trains and saves a checkpoint.
- The RL loop completes without shape, reward, or environment errors.
- Deterministic evaluation returns a complete metric dictionary.
- Monte Carlo evaluation returns both summary statistics and per-simulation diagnostics.

What not to over-interpret from this notebook:

- A small positive return does not prove the strategy is good.
- A negative result does not necessarily mean the implementation is wrong.
- With very few episodes and simulations, this notebook is best treated as a systems test.

Good next steps after the smoke test:

1. Increase `num_episodes`.
2. Increase `monte_carlo_num_simulations`.
3. Compare `algo="ppo"` versus `algo="sac"`.
4. Run the walk-forward pipeline for multiple folds.
5. Add your own asset universe and date ranges.